# Swin geo context ablation

Loads the completed fold0 geo checkpoint and evaluates validation RMSE while zeroing context feature groups. No retraining.


In [ ]:
from pathlib import Path
import subprocess
import sys


def install_p100_torch_if_needed():
    try:
        gpu = subprocess.check_output(
            ['bash', '-lc', "nvidia-smi --query-gpu=name --format=csv,noheader | head -1"],
            text=True,
        ).strip()
    except Exception:
        gpu = ''
    if 'P100' in gpu:
        print(f'Installing a Pascal-compatible PyTorch build for {gpu}')
        subprocess.run(
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '-q',
                '--force-reinstall',
                'torch==2.5.1',
                'torchvision==0.20.1',
                '--index-url',
                'https://download.pytorch.org/whl/cu121',
            ],
            check=True,
        )
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'pillow==11.3.0'],
            check=True,
        )
    return gpu


gpu_name = install_p100_torch_if_needed()

REPO = Path('/kaggle/working/solafune-nowcast')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/shionsuio/solafune-nowcast.git', str(REPO)], check=True)

sys.path.insert(0, str(REPO / 'src'))

import torch
print('GPU:', gpu_name or (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'), 'torch:', torch.__version__)

from evaluate_geo_context_ablation import run

DATA_ROOT = Path('/kaggle/input/solafune-dataset-v2')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('/kaggle/input/datasets/suioshion/solafune-dataset-v2')

def find_first(pattern, preferred_text):
    candidates = sorted(Path('/kaggle/input').rglob(pattern))
    print(pattern, 'candidates:')
    for path in candidates[:30]:
        print(' ', path)
    preferred = [path for path in candidates if preferred_text in str(path)]
    if preferred:
        return preferred[0]
    if candidates:
        return candidates[0]
    raise FileNotFoundError(pattern)


CHECKPOINT_ROOT = Path('/kaggle/working/geo_checkpoint')
CHECKPOINT_ROOT.mkdir(exist_ok=True)
best_path = find_first('best_fold0.pth', 'solafune-swin-temporal-geo')
stats_path = find_first('band_stats_fold0.json', 'solafune-swin-temporal-geo')
(CHECKPOINT_ROOT / 'best_fold0.pth').write_bytes(best_path.read_bytes())
(CHECKPOINT_ROOT / 'band_stats_fold0.json').write_bytes(stats_path.read_bytes())
print('CHECKPOINT_ROOT:', CHECKPOINT_ROOT)

LOCATION_METADATA = REPO / 'data' / 'location_coordinates_manual.csv'


class Args:
    root = '/kaggle/working'
    kaggle_input_root = str(DATA_ROOT)
    checkpoint_root = str(CHECKPOINT_ROOT)
    fold = 0
    modes = 'all'
    batch_size = 16
    workers = 2
    model_subdir = 'swin_v2_temporal_geo'
    location_metadata_path = str(LOCATION_METADATA)
    output = 'outputs/swin_v2_temporal_geo/context_ablation_fold0.csv'
    no_amp = True


summary_path = run(Args())

import pandas as pd
summary = pd.read_csv(summary_path).sort_values('validation_rmse')
display(summary)
